# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [14]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.127658084030336)),
 (1, np.float64(2.127658084030336)),
 (2, np.float64(2.127658084030336)),
 (3, np.float64(2.127658084030336)),
 (4, np.float64(2.127658084030336))]

## Inverted index

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('it', 18)]),
 (1, [('a', 2), ('is', 18), ('what', 10)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.03597172435034013)),
   (None, np.float64(0.061985035252107745)),
   (None, np.float64(0.06665699816571602)),
   (None, np.float64(0.0722949331308399)),
   (None, np.float64(0.0804471556831593)),
   (None, np.float64(0.13205017610833358)),
   (None, np.float64(0.15306000981189893)),
   (None, np.float64(0.15895915589179832)),
   (None, np.float64(0.1768128363533965)),
   (None, np.float64(0.1797868530297817)),
   (None, np.float64(0.21642432105938747)),
   (None, np.float64(0.2416682847982542)),
   (None, np.float64(0.29486074001144846)),
   (None, np.float64(0.3136395402001759)),
   (None, np.float64(0.4124163324589566)),
   (None, np.float64(0.4573742928060448)),
   (None, np.float64(0.46505031395533714)),
   (None, np.float64(0.475947431733418))]),
 (1,
  [(None, np.float64(0.5061770456723188)),
   (None, np.float64(0.509795020066186)),
   (None, np.float64(0.6013187505475723)),
   (None, np.float64(0.6156108045265163)),
   (None, np.float64(0.65014486955

In [21]:
from typing import Iterator
import numpy as np
from collections import defaultdict

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(
        map(
            lambda x: REDUCE(*x),
            groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))
        )
    )

def groupbykey_distributed(map_partitions, PARTITIONER):
    global reducers
    partitions = [dict() for _ in range(reducers)]
    for map_partition in map_partitions:
        for (k2, v2) in map_partition:
            p = partitions[PARTITIONER(k2)]
            p[k2] = p.get(k2, []) + [v2]
    return [
        (partition_id, sorted(partition.items(), key=lambda x: x[0]))
        for (partition_id, partition) in enumerate(partitions)
    ]

def PARTITIONER(obj):
    global reducers
    return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
    map_partitions = map(
        lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)),
        INPUTFORMAT()
    )
    if COMBINER is not None:
        map_partitions = map(
            lambda map_partition: flatten(
                map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))
            ),
            map_partitions
        )
    reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER)
    reduce_outputs = map(
        lambda reduce_partition: (
            reduce_partition[0],
            flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))
        ),
        reduce_partitions
    )
    print("{} key-value pairs were sent over a network.".format(
        sum(len(vs) for (k, vs) in flatten([partition for (partition_id, partition) in reduce_partitions]))
    ))
    return reduce_outputs

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [22]:
numbers = [7, 3, 11, -2, 11, 5, 9]

def RECORDREADER():
    for idx, x in enumerate(numbers):
        yield (idx, x)

def MAP(k1, v1):
    yield ("max", v1)

def REDUCE(k2, values):
    yield (k2, max(values))

list(MapReduce(RECORDREADER, MAP, REDUCE))

[('max', 11)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [23]:
numbers = [7, 3, 11, -2, 11, 5, 9]

def RECORDREADER():
    for idx, x in enumerate(numbers):
        yield (idx, x)

def MAP(k1, v1):
    yield ("avg", (v1, 1))

def REDUCE(k2, values):
    total = 0
    count = 0
    for x, c in values:
        total += x
        count += c
    yield (k2, total / count if count else 0)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[('avg', 6.285714285714286)]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [24]:
def groupbykey_sorted(iterable):
    data = sorted(iterable, key=lambda x: x[0])
    if not data:
        return []
    result = []
    current_key = data[0][0]
    current_values = []
    for k, v in data:
        if k != current_key:
            result.append((current_key, current_values))
            current_key = k
            current_values = [v]
        else:
            current_values.append(v)
    result.append((current_key, current_values))
    return result

example = [("b", 2), ("a", 1), ("b", 3), ("a", 5), ("c", 9)]
groupbykey_sorted(example)

[('a', [1, 5]), ('b', [2, 3]), ('c', [9])]

### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [25]:
input_values = [1, 2, 2, 3, 1, 4, 5, 5, 2, 6, 4, 7]
maps = 3
reducers = 2

def INPUTFORMAT():
    global maps
    split_size = int(np.ceil(len(input_values) / maps))
    def RECORDREADER(split):
        for idx, x in enumerate(split):
            yield (idx, x)
    for i in range(0, len(input_values), split_size):
        yield RECORDREADER(input_values[i:i+split_size])

def MAP(k1, v1):
    yield (v1, None)

def REDUCE(k2, values):
    yield (k2, None)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=REDUCE)
distinct_values = sorted([k for partition_id, part in partitioned_output for (k, _) in part])
distinct_values

10 key-value pairs were sent over a network.


[1, 2, 3, 4, 5, 6, 7]

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [26]:
R = [
    (1, "Иван", "IT"),
    (2, "Анна", "HR"),
    (3, "Пётр", "IT"),
    (4, "Ольга", "FIN"),
]

S = [
    (3, "Пётр", "IT"),
    (4, "Ольга", "FIN"),
    (5, "Мария", "MKT"),
]

employees = [
    (1, "IT", 120),
    (2, "HR", 90),
    (3, "IT", 150),
    (4, "FIN", 110),
]
def RECORDREADER():
    for idx, t in enumerate(R):
        yield (idx, t)

def MAP(k1, t):
    if t[2] == "IT":
        yield (t, t)

def REDUCE(t, values):
    for v in values:
        yield (t, v)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[((1, 'Иван', 'IT'), (1, 'Иван', 'IT')),
 ((3, 'Пётр', 'IT'), (3, 'Пётр', 'IT'))]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [27]:
def RECORDREADER():
    for idx, t in enumerate(R):
        yield (idx, t)

def MAP(k1, t):
    t_prime = (t[1], t[2])  # имя, отдел
    yield (t_prime, t_prime)

def REDUCE(t_prime, values):
    yield (t_prime, t_prime)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[(('Иван', 'IT'), ('Иван', 'IT')),
 (('Анна', 'HR'), ('Анна', 'HR')),
 (('Пётр', 'IT'), ('Пётр', 'IT')),
 (('Ольга', 'FIN'), ('Ольга', 'FIN'))]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [28]:
union_input = [("R", t) for t in R] + [("S", t) for t in S]

def RECORDREADER():
    for idx, (_, t) in enumerate(union_input):
        yield (idx, t)

def MAP(k1, t):
    yield (t, t)

def REDUCE(t, values):
    yield (t, t)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[((1, 'Иван', 'IT'), (1, 'Иван', 'IT')),
 ((2, 'Анна', 'HR'), (2, 'Анна', 'HR')),
 ((3, 'Пётр', 'IT'), (3, 'Пётр', 'IT')),
 ((4, 'Ольга', 'FIN'), (4, 'Ольга', 'FIN')),
 ((5, 'Мария', 'MKT'), (5, 'Мария', 'MKT'))]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [29]:
intersection_input = [("R", t) for t in R] + [("S", t) for t in S]

def RECORDREADER():
    for idx, (_, t) in enumerate(intersection_input):
        yield (idx, t)

def MAP(k1, t):
    yield (t, t)

def REDUCE(t, values):
    vals = list(values)
    if len(vals) >= 2:
        yield (t, t)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[((3, 'Пётр', 'IT'), (3, 'Пётр', 'IT')),
 ((4, 'Ольга', 'FIN'), (4, 'Ольга', 'FIN'))]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [30]:
difference_input = [("R", t) for t in R] + [("S", t) for t in S]

def RECORDREADER():
    for idx, (rel, t) in enumerate(difference_input):
        yield (idx, (rel, t))

def MAP(k1, rel_tuple):
    rel, t = rel_tuple
    yield (t, rel)

def REDUCE(t, rels):
    rels = list(rels)
    if rels == ["R"]:
        yield (t, t)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[((1, 'Иван', 'IT'), (1, 'Иван', 'IT')),
 ((2, 'Анна', 'HR'), (2, 'Анна', 'HR'))]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [31]:
# R1(A,B), S1(B,C)
R1 = [("a1", "b1"), ("a2", "b2"), ("a3", "b1")]
S1 = [("b1", "c1"), ("b1", "c2"), ("b3", "c3")]

join_input = [("R", t) for t in R1] + [("S", t) for t in S1]

def RECORDREADER():
    for idx, item in enumerate(join_input):
        yield (idx, item)

def MAP(k1, tagged_tuple):
    rel, t = tagged_tuple
    if rel == "R":
        a, b = t
        yield (b, ("R", a))
    else:
        b, c = t
        yield (b, ("S", c))

def REDUCE(b, values):
    r_vals = []
    s_vals = []
    for rel, x in values:
        if rel == "R":
            r_vals.append(x)
        else:
            s_vals.append(x)
    for a in r_vals:
        for c in s_vals:
            yield (None, (a, b, c))

list(MapReduce(RECORDREADER, MAP, REDUCE))

[(None, ('a1', 'b1', 'c1')),
 (None, ('a1', 'b1', 'c2')),
 (None, ('a3', 'b1', 'c1')),
 (None, ('a3', 'b1', 'c2'))]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [32]:
def RECORDREADER():
    for idx, t in enumerate(employees):
        yield (idx, t)

def MAP(k1, t):
    emp_id, dept, score = t
    yield (dept, score)

def REDUCE(dept, scores):
    yield (dept, max(scores))

list(MapReduce(RECORDREADER, MAP, REDUCE))

[('IT', 150), ('HR', 90), ('FIN', 110)]

#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [33]:
np.random.seed(7)
I, J, K = 2, 3, 8
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def RECORDREADER():
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
    j, k = k1
    w = v1
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
    yield (key, sum(values))

def asmatrix(reduce_output):
    reduce_output = list(reduce_output)
    I = max(i for ((i, k), vw) in reduce_output) + 1
    K = max(k for ((i, k), vw) in reduce_output) + 1
    mat = np.empty(shape=(I, K))
    for ((i, k), vw) in reduce_output:
        mat[i, k] = vw
    return mat

reference_solution = np.matmul(small_mat, big_mat)
solution = asmatrix(MapReduce(RECORDREADER, MAP, REDUCE))
np.allclose(reference_solution, solution)

True

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [34]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [35]:
np.random.seed(7)
I, J, K = 2, 3, 8
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def RECORDREADER():
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
    j, k = k1
    w = v1
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
    yield (key, sum(values))

def asmatrix(reduce_output):
    reduce_output = list(reduce_output)
    I = max(i for ((i, k), vw) in reduce_output) + 1
    K = max(k for ((i, k), vw) in reduce_output) + 1
    mat = np.empty(shape=(I, K))
    for ((i, k), vw) in reduce_output:
        mat[i, k] = vw
    return mat

reference_solution = np.matmul(small_mat, big_mat)
solution = asmatrix(MapReduce(RECORDREADER, MAP, REDUCE))
np.allclose(reference_solution, solution)

True

Проверьте своё решение

In [36]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [37]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [38]:
np.random.seed(11)
M = np.random.rand(2, 3)
N = np.random.rand(3, 4)

def matrix_multiply_both_one_machine(M, N):
    # Этап 1: join по j
    def RECORDREADER_1():
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                yield (("M", i, j), M[i, j])
        for j in range(N.shape[0]):
            for k in range(N.shape[1]):
                yield (("N", j, k), N[j, k])

    def MAP_1(k1, v1):
        tag = k1[0]
        if tag == "M":
            _, i, j = k1
            yield (j, ("M", i, v1))
        else:
            _, j, k = k1
            yield (j, ("N", k, v1))

    def REDUCE_1(j, values):
        left = []
        right = []
        for item in values:
            if item[0] == "M":
                left.append(item[1:])
            else:
                right.append(item[1:])
        for i, v in left:
            for k, w in right:
                yield ((i, k), v * w)

    stage1 = list(MapReduce(RECORDREADER_1, MAP_1, REDUCE_1))

    # Этап 2: суммирование частичных произведений по (i, k)
    def RECORDREADER_2():
        for idx, pair in enumerate(stage1):
            yield (idx, pair)

    def MAP_2(k1, v1):
        key, value = v1
        yield (key, value)

    def REDUCE_2(key, values):
        yield (key, sum(values))

    return asmatrix(MapReduce(RECORDREADER_2, MAP_2, REDUCE_2))

solution = matrix_multiply_both_one_machine(M, N)
np.allclose(solution, np.matmul(M, N))

True

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [39]:
np.random.seed(21)
M = np.random.rand(2, 3)
N = np.random.rand(3, 5)
maps = 2
reducers = 3

def matrix_multiply_distributed_two_readers(M, N):
    global reducers

    def INPUTFORMAT_1():
        def READ_M():
            for i in range(M.shape[0]):
                for j in range(M.shape[1]):
                    yield (("M", i, j), M[i, j])

        def READ_N():
            for j in range(N.shape[0]):
                for k in range(N.shape[1]):
                    yield (("N", j, k), N[j, k])

        yield READ_M()
        yield READ_N()

    def MAP_1(k1, v1):
        tag = k1[0]
        if tag == "M":
            _, i, j = k1
            yield (j, ("M", i, v1))
        else:
            _, j, k = k1
            yield (j, ("N", k, v1))

    def REDUCE_1(j, values):
        left = []
        right = []
        for item in values:
            if item[0] == "M":
                left.append(item[1:])
            else:
                right.append(item[1:])
        for i, v in left:
            for k, w in right:
                yield ((i, k), v * w)

    def PARTITIONER_J(j):
        return hash(j) % reducers

    stage1 = [
        pair
        for partition_id, partition in MapReduceDistributed(
            INPUTFORMAT_1, MAP_1, REDUCE_1, PARTITIONER=PARTITIONER_J
        )
        for pair in partition
    ]

    def INPUTFORMAT_2():
        split_size = int(np.ceil(len(stage1) / maps))
        def READ_PART(split):
            for idx, item in enumerate(split):
                yield (idx, item)
        for start in range(0, len(stage1), split_size):
            yield READ_PART(stage1[start:start + split_size])

    def MAP_2(k1, v1):
        key, value = v1
        yield (key, value)

    def REDUCE_2(key, values):
        yield (key, sum(values))

    def PARTITIONER_IK(key):
        return hash(key) % reducers

    stage2 = [
        pair
        for partition_id, partition in MapReduceDistributed(
            INPUTFORMAT_2, MAP_2, REDUCE_2, PARTITIONER=PARTITIONER_IK, COMBINER=REDUCE_2
        )
        for pair in partition
    ]
    return asmatrix(stage2)

solution = matrix_multiply_distributed_two_readers(M, N)
np.allclose(solution, np.matmul(M, N))

21 key-value pairs were sent over a network.
20 key-value pairs were sent over a network.


True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [40]:
np.random.seed(33)
M = np.random.rand(3, 4)
N = np.random.rand(4, 2)
maps = 4
reducers = 3

def split_matrix_M(M, chunks):
    rows_per_chunk = int(np.ceil(M.shape[0] / chunks))
    readers = []
    for start in range(0, M.shape[0], rows_per_chunk):
        stop = min(start + rows_per_chunk, M.shape[0])
        def reader(start=start, stop=stop):
            for i in range(start, stop):
                for j in range(M.shape[1]):
                    yield (("M", i, j), M[i, j])
        readers.append(reader())
    return readers

def split_matrix_N(N, chunks):
    cols_per_chunk = int(np.ceil(N.shape[1] / chunks))
    readers = []
    for start in range(0, N.shape[1], cols_per_chunk):
        stop = min(start + cols_per_chunk, N.shape[1])
        def reader(start=start, stop=stop):
            for j in range(N.shape[0]):
                for k in range(start, stop):
                    yield (("N", j, k), N[j, k])
        readers.append(reader())
    return readers

def matrix_multiply_distributed_many_readers(M, N):
    global reducers

    def INPUTFORMAT_1():
        for rr in split_matrix_M(M, 2):
            yield rr
        for rr in split_matrix_N(N, 2):
            yield rr

    def MAP_1(k1, v1):
        tag = k1[0]
        if tag == "M":
            _, i, j = k1
            yield (j, ("M", i, v1))
        else:
            _, j, k = k1
            yield (j, ("N", k, v1))

    def REDUCE_1(j, values):
        left = []
        right = []
        for item in values:
            if item[0] == "M":
                left.append(item[1:])
            else:
                right.append(item[1:])
        for i, v in left:
            for k, w in right:
                yield ((i, k), v * w)

    stage1 = [
        pair
        for partition_id, partition in MapReduceDistributed(INPUTFORMAT_1, MAP_1, REDUCE_1)
        for pair in partition
    ]

    def INPUTFORMAT_2():
        split_size = int(np.ceil(len(stage1) / maps))
        def READ_PART(split):
            for idx, item in enumerate(split):
                yield (idx, item)
        for start in range(0, len(stage1), split_size):
            yield READ_PART(stage1[start:start + split_size])

    def MAP_2(k1, v1):
        key, value = v1
        yield (key, value)

    def REDUCE_2(key, values):
        yield (key, sum(values))

    stage2 = [
        pair
        for partition_id, partition in MapReduceDistributed(
            INPUTFORMAT_2, MAP_2, REDUCE_2, COMBINER=REDUCE_2
        )
        for pair in partition
    ]
    return asmatrix(stage2)

solution = matrix_multiply_distributed_many_readers(M, N)
np.allclose(solution, np.matmul(M, N))

20 key-value pairs were sent over a network.
24 key-value pairs were sent over a network.


True